# Titanic with Spark MLlib

In [1]:
import warnings
warnings.filterwarnings('ignore')
spark_ui_port = 4040
app_name = "Otus"

from itertools import groupby

In [2]:
import pyspark

spark = (
    pyspark.sql.SparkSession.builder
        .appName(app_name)
        .master("local[4]") # limit executor to 4 cores
        .config("spark.executor.memory", "1g")
        .config("spark.driver.memory", "1g")
        .config("spark.ui.port", spark_ui_port)
        .getOrCreate()
)
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)  # to pretty print pyspark.DataFrame in jupyter

# sc = spark.sparkContext
# sc.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/05 20:32:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv("./titanic/train.csv", inferSchema=True, header=True)
df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [4]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

In [5]:
# dataset shape
(len(df.columns), df.count())

(12, 891)

А как получить строки с 5 по 12?

В PySpark нетпрямой индексации строк как в Pandas через iloc

In [6]:
# прямой путь - указать номера строк

df.limit(12).subtract(df.limit(4)).show()

# df.limit(12) - получает первые 12 строк
# .subtract(df.limit(4)) - вычисляет разность между вызовом (df.limit(12)) и аргументом (df.limit(4)) - т.е получим первые 12 строк и потом от них "вычтем" первые 4 - получим с 5 по 12

+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch| Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+
|          7|       0|     1|McCarthy, Mr. Tim...|  male|54.0|    0|    0|  17463|51.8625|  E46|       S|
|         11|       1|     3|Sandstrom, Miss. ...|female| 4.0|    1|    1|PP 9549|   16.7|   G6|       S|
|         12|       1|     1|Bonnell, Miss. El...|female|58.0|    0|    0| 113783|  26.55| C103|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0| 373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0| 330877| 8.4583| NULL|       Q|
|          8|       0|     3|Palsson, Master. ...|  male| 2.0|    3|    1| 349909| 21.075| NULL|       S|
|          9|       1|     3|Johnson, Mrs. Osc

In [7]:
# Создадим дополнительный сквозной индекс и по нему отфильтруем строки

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number # создает последовательный номер - используем для построения индекса

w = Window.orderBy('PassengerId')
df_w_idx = df.withColumn('row_num', row_number().over(w))
df_w_idx.filter((df_w_idx.row_num > 4) & (df_w_idx.row_num < 12)).show()

+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+-------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch| Ticket|   Fare|Cabin|Embarked|row_num|
+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+-------+
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0| 373450|   8.05| NULL|       S|      5|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0| 330877| 8.4583| NULL|       Q|      6|
|          7|       0|     1|McCarthy, Mr. Tim...|  male|54.0|    0|    0|  17463|51.8625|  E46|       S|      7|
|          8|       0|     3|Palsson, Master. ...|  male| 2.0|    3|    1| 349909| 21.075| NULL|       S|      8|
|          9|       1|     3|Johnson, Mrs. Osc...|female|27.0|    0|    2| 347742|11.1333| NULL|       S|      9|
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0| 237736|30.070

25/09/05 20:33:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/05 20:33:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/05 20:33:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [8]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



Так смотреть не удобно - отформатируем вывод

In [11]:
dt = df.dtypes
for r in dt:
    print(f"{r[0]:>17}\t{r[1]}")

      PassengerId	int
         Survived	int
           Pclass	int
             Name	string
              Sex	string
              Age	double
            SibSp	int
            Parch	int
           Ticket	string
             Fare	double
            Cabin	string
         Embarked	string


Давайте cортируем по типам

In [12]:
for r in sorted(
    df.dtypes, 
    key=lambda x: x[1]
):
    print(f"{r[0]:>17}\t{r[1]}")

              Age	double
             Fare	double
      PassengerId	int
         Survived	int
           Pclass	int
            SibSp	int
            Parch	int
             Name	string
              Sex	string
           Ticket	string
            Cabin	string
         Embarked	string


Соберем по типам

In [13]:
dt.sort(key=lambda x: x[1])

print('Data types:')
for k, g in groupby(dt, lambda x: x[1]):
    print(f'{k:<6} - {len(list(g))}')

Data types:
double - 2
int    - 5
string - 5


## Кодирование категориальных признаков

In [14]:
df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [15]:
from pyspark.ml.feature import StringIndexer

### Sex

<div class="alert alert-warning">

<b>Warning! Частая ошибка</b>


    
StringIndexer может работать с пропущенными значениями только в формате <b>NaN</b>, но не <b>NULL</b>!

<b>Необходимо проверить на пропуски!</b>

Либо указать ключ `handleInvalid='keep'`
</div>

In [16]:
df.filter(df['Sex'].isNull()).count()

0

In [18]:
sex_indexer = StringIndexer(handleInvalid='keep')\
    .setInputCol('Sex')\
    .setOutputCol('SexIndexed')
df_prep = sex_indexer.fit(df).transform(df)
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndexed|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|       0.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|       1.0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|       1.0|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|       1.0|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|    

In [19]:
sex_indexer = StringIndexer(inputCol="Sex", outputCol="SexIndex", handleInvalid='keep')
sex_indexer_model = sex_indexer.fit(df)
df_prep = sex_indexer_model.transform(df)
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|     1.0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|     1.0|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|     1.0|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|     0.0|
+-------

### Pclass

In [21]:
from pyspark.ml.feature import OneHotEncoder

pclass_indexer = StringIndexer(handleInvalid='keep')\
    .setInputCol('Pclass')\
    .setOutputCol('PclassIndex')

pclass_indexer_model = pclass_indexer.fit(df_prep)
df_prep = pclass_indexer_model.transform(df_prep)
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|     1.0|        1.0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|     1.0|        0.0|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|     1.0|        1.0|
|          5|       0|     3|Allen, Mr. Willia..

In [22]:
pclass_encoder = OneHotEncoder(handleInvalid='keep')\
    .setInputCol('PclassIndex')\
    .setOutputCol('PclassEncoded')

pclass_encoder_model = pclass_encoder.fit(df_prep)
df_prep = pclass_encoder_model.transform(df_prep)
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|     1.0|        1.0|(4,[1],[1.0])|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|     1.0|        0.0|(4,[0],[1.0])|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| 

In [23]:
df_prep.head(5)

[Row(PassengerId=1, Survived=0, Pclass=3, Name='Braund, Mr. Owen Harris', Sex='male', Age=22.0, SibSp=1, Parch=0, Ticket='A/5 21171', Fare=7.25, Cabin=None, Embarked='S', SexIndex=0.0, PclassIndex=0.0, PclassEncoded=SparseVector(4, {0: 1.0})),
 Row(PassengerId=2, Survived=1, Pclass=1, Name='Cumings, Mrs. John Bradley (Florence Briggs Thayer)', Sex='female', Age=38.0, SibSp=1, Parch=0, Ticket='PC 17599', Fare=71.2833, Cabin='C85', Embarked='C', SexIndex=1.0, PclassIndex=1.0, PclassEncoded=SparseVector(4, {1: 1.0})),
 Row(PassengerId=3, Survived=1, Pclass=3, Name='Heikkinen, Miss. Laina', Sex='female', Age=26.0, SibSp=0, Parch=0, Ticket='STON/O2. 3101282', Fare=7.925, Cabin=None, Embarked='S', SexIndex=1.0, PclassIndex=0.0, PclassEncoded=SparseVector(4, {0: 1.0})),
 Row(PassengerId=4, Survived=1, Pclass=1, Name='Futrelle, Mrs. Jacques Heath (Lily May Peel)', Sex='female', Age=35.0, SibSp=1, Parch=0, Ticket='113803', Fare=53.1, Cabin='C123', Embarked='S', SexIndex=1.0, PclassIndex=1.0, Pc

In [24]:
df_prep.head()[-1]

SparseVector(4, {0: 1.0})

In [26]:
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|     1.0|        1.0|(4,[1],[1.0])|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|     1.0|        0.0|(4,[0],[1.0])|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| 

### Embarked

In [21]:
df_prep.filter(df_prep['Embarked'].isNull()).count()

2

<div class="alert alert-warning">

<b>Warning! Частая ошибка</b>



StringIndexer может работать с пропущенными значениями только в формате <b>NaN</b>, но не <b>NULL</b>!

Если мы закодируем значения `Embarked` то мы не увидим ошибки. Мы получим ошибку только при обращении к этой строке!
    
</div>

In [27]:
embark_indexer = StringIndexer()\
    .setInputCol('Embarked')\
    .setOutputCol('EmbarkedIndex')

df_err = embark_indexer.fit(df_prep).transform(df_prep)
df_err.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|EmbarkedIndex|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|          0.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|     1.0|        1.0|(4,[1],[1.0])|          1.0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|     1.0|        0.0|(4,[0],[1.0])|          0.0|
|          4|   

In [ ]:
# uncomment for error

# df_err.show(62)

25/09/05 13:13:21 ERROR Executor: Exception in task 0.0 in stage 53.0 (TID 35)
org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] User defined function (`StringIndexerModel$$Lambda$4476/0x00000090022190f0`: (string) => double) failed due to: org.apache.spark.SparkException: StringIndexer encountered NULL value. To handle or skip NULLS, try setting StringIndexer.handleInvalid.. SQLSTATE: 39000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:195)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvalu

Py4JJavaError: An error occurred while calling o328.showString.
: org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] User defined function (`StringIndexerModel$$Lambda$4476/0x00000090022190f0`: (string) => double) failed due to: org.apache.spark.SparkException: StringIndexer encountered NULL value. To handle or skip NULLS, try setting StringIndexer.handleInvalid.. SQLSTATE: 39000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:195)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.s

In [40]:
embark_indexer = StringIndexer(handleInvalid='keep')\
    .setInputCol('Embarked')\
    .setOutputCol('EmbarkedIndex')

df_err = embark_indexer.fit(df_prep).transform(df_prep)
df_err.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|EmbarkedIndex|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|          0.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|     1.0|        1.0|(4,[1],[1.0])|          1.0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|     1.0|        0.0|(4,[0],[1.0])|          0.0|
|          4|   

In [41]:
df_err.show(62)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+--------+-----------+--------+--------+-----------+-------------+-------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|    Fare|      Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|EmbarkedIndex|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+--------+-----------+--------+--------+-----------+-------------+-------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|    7.25|       NULL|       S|     0.0|        0.0|(4,[0],[1.0])|          0.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599| 71.2833|        C85|       C|     1.0|        1.0|(4,[1],[1.0])|          1.0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|   7.925|       NULL|       S|     1.0|        0.0|(4,

#### Заполним пропуски

In [42]:
df_prep = df_prep.fillna('X', subset=['Embarked'])
df_prep.show(62)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+--------+-----------+--------+--------+-----------+-------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|    Fare|      Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+--------+-----------+--------+--------+-----------+-------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|    7.25|       NULL|       S|     0.0|        0.0|(4,[0],[1.0])|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599| 71.2833|        C85|       C|     1.0|        1.0|(4,[1],[1.0])|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|   7.925|       NULL|       S|     1.0|        0.0|(4,[0],[1.0])|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|3

In [43]:
embark_indexer = StringIndexer()\
    .setInputCol('Embarked')\
    .setOutputCol('EmbarkedIndex')

embark_indexer_model = embark_indexer.fit(df_prep)
df_prep = embark_indexer_model.transform(df_prep)
df_prep.show(62)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+--------+-----------+--------+--------+-----------+-------------+-------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|    Fare|      Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|EmbarkedIndex|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+--------+-----------+--------+--------+-----------+-------------+-------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|    7.25|       NULL|       S|     0.0|        0.0|(4,[0],[1.0])|          0.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599| 71.2833|        C85|       C|     1.0|        1.0|(4,[1],[1.0])|          1.0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|   7.925|       NULL|       S|     1.0|        0.0|(4,

In [44]:
embarked_encoder = OneHotEncoder()\
    .setInputCol('EmbarkedIndex')\
    .setOutputCol('EmbarkedEncoded')

embarked_encoder_model = embarked_encoder.fit(df_prep)
df_prep = embarked_encoder_model.transform(df_prep)
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|EmbarkedIndex|EmbarkedEncoded|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|          0.0|  (3,[0],[1.0])|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|     1.0|        1.0|(4,[1],[1.0])|          1.0|  (3,[1],[1.0])|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| 

## Нормализация числовых признаков

### Age

In [45]:
from pyspark.ml.feature import Imputer

In [47]:
age_imputer = Imputer()\
    .setInputCols(['Age'])\
    .setOutputCols(['AgeImputed'])\
    .setStrategy('mean')


age_imputer_model = age_imputer.fit(df_prep)
df_prep = age_imputer_model.transform(df_prep)
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|EmbarkedIndex|EmbarkedEncoded|AgeImputed|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|          0.0|  (3,[0],[1.0])|      22.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|     1.0|        1.0|(4,[1],[1.0])|          1.0|  (3,[1],[1.0])|      38.0|
|          3|       1|     3|Heikkinen, Miss.

In [48]:
from pyspark.ml.feature import VectorAssembler

In [54]:
age_assembler = VectorAssembler()\
    .setInputCols(["AgeImputed"])\
    .setOutputCol("AgeVector")

df_prep = age_assembler.transform(df_prep)
df_prep.show(5)

IllegalArgumentException: Output column AgeVector already exists.

In [50]:
from pyspark.ml.feature import MinMaxScaler

In [51]:
age_scaler = MinMaxScaler()\
    .setInputCol('AgeVector')\
    .setOutputCol('AgeScaled')

age_scaler_model = age_scaler.fit(df_prep)
df_prep = age_scaler_model.transform(df_prep)
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+----------+---------+--------------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|EmbarkedIndex|EmbarkedEncoded|AgeImputed|AgeVector|           AgeScaled|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+----------+---------+--------------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|          0.0|  (3,[0],[1.0])|      22.0|   [22.0]|[0.2711736617240513]|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       

In [52]:
df_prep.filter(df_prep['Fare'].isNull()).count()

0

In [53]:
fare_imputer = Imputer(inputCol="Fare", outputCol="FareImputed")
fare_imputer_model = fare_imputer.fit(df_prep)
df_prep = fare_imputer_model.transform(df_prep)
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+----------+---------+--------------------+-----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|EmbarkedIndex|EmbarkedEncoded|AgeImputed|AgeVector|           AgeScaled|FareImputed|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+----------+---------+--------------------+-----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|          0.0|  (3,[0],[1.0])|      22.0|   [22.0]|[0.2711736617240513]|       7.25|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|  

In [55]:
fare_assembler = VectorAssembler(inputCols=["FareImputed"], outputCol="FareVector")
df_prep = fare_assembler.transform(df_prep)
df_prep.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+----------+---------+--------------------+-----------+----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|EmbarkedIndex|EmbarkedEncoded|AgeImputed|AgeVector|           AgeScaled|FareImputed|FareVector|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+-------------+---------------+----------+---------+--------------------+-----------+----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|          0.0|  (3,[0],[1.0])|      22.0|   [22.0]|[0.2711736617240513]|       7.25|    [7.25]|
|          2|       

`pyspark.ml.feature.RobustScaler` — это трансформер для масштабирования числовых признаков, устойчивый к выбросам.

В отличие от стандартного `StandardScaler` (который использует среднее и стандартное отклонение), `RobustScaler` нормализует данные, используя:
	•	медиану (центрует данные относительно медианы)
	•	интерквартильный размах (IQR = Q3 − Q1) для масштабирования

То есть каждое значение x преобразуется так:

$$x’ = \frac{x - \text{median}(X)}{\text{IQR}(X)}$$

где X — значения в колонке.

In [57]:
from pyspark.ml.feature import RobustScaler

fare_scaler = RobustScaler(inputCol="FareVector", outputCol="FareScaled")
fare_scaler_model = fare_scaler.fit(df_prep)
df_prep = fare_scaler_model.transform(df_prep)
df_prep.show(5)


IllegalArgumentException: requirement failed: Output column FareScaled already exists.

## Собираем вектор признаков

Для алгоритмов МО из Spark MlLib нужно подавать на вход столбец с вектором признаков

In [58]:
from pyspark.ml.feature import VectorAssembler

features_assembler = VectorAssembler(inputCols=[
    "SexIndex",
    "PclassEncoded",
    "AgeScaled",
    "FareScaled",
    ],
    outputCol="Features",
)

prep_df = df
prep_df = sex_indexer_model.transform(prep_df)
prep_df = pclass_indexer_model.transform(prep_df)
prep_df = pclass_encoder_model.transform(prep_df)
prep_df = age_imputer_model.transform(prep_df)
prep_df = age_assembler.transform(prep_df)
prep_df = age_scaler_model.transform(prep_df)
prep_df = fare_imputer_model.transform(prep_df)
prep_df = fare_assembler.transform(prep_df)
prep_df = fare_scaler_model.transform(prep_df)
feat_df = features_assembler.transform(prep_df)

feat_df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+----------+---------+--------------------+-----------+----------+--------------------+--------------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|AgeImputed|AgeVector|           AgeScaled|FareImputed|FareVector|          FareScaled|            Features|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+----------+---------+--------------------+-----------+----------+--------------------+--------------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|      22.0|   [22.0]|[0.2711736617240513]|       7.25|    [7.25]|[0.313795760078

## Конвейер

Объединим различные этапы подготовки признаков в единый конвейер

In [59]:
from pyspark.ml.pipeline import Pipeline

feat_ext_pipe = Pipeline(stages=[
    sex_indexer,
    pclass_indexer,
    pclass_encoder,
    age_imputer,
    age_assembler,
    age_scaler,
    fare_imputer_model,
    fare_assembler,
    fare_scaler_model,
    features_assembler,
]).fit(df)


In [60]:
feat_df = feat_ext_pipe.transform(df)
feat_df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+----------+---------+--------------------+-----------+----------+--------------------+--------------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|AgeImputed|AgeVector|           AgeScaled|FareImputed|FareVector|          FareScaled|            Features|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+-----------+-------------+----------+---------+--------------------+-----------+----------+--------------------+--------------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     0.0|        0.0|(4,[0],[1.0])|      22.0|   [22.0]|[0.2711736617240513]|       7.25|    [7.25]|[0.313795760078

## Сохранение

Сохраним конвейер на диск для последующего использования при подготовке других данных

In [61]:
feat_ext_pipe.write().overwrite().save(f"{app_name}_feat_exty_pipe")

## Обработка тестовых данных

In [62]:
from pyspark.ml.pipeline import PipelineModel

test_df = spark.read.csv("./titanic/test.csv", inferSchema=True, header=True)

feat_ext_pipe_loaded = PipelineModel.load(f"{app_name}_feat_exty_pipe")

prep_test_df = feat_ext_pipe_loaded.transform(test_df)
prep_test_df.show(5)

+-----------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+--------+-----------+-------------+----------+---------+--------------------+-----------+----------+--------------------+--------------------+
|PassengerId|Pclass|                Name|   Sex| Age|SibSp|Parch| Ticket|   Fare|Cabin|Embarked|SexIndex|PclassIndex|PclassEncoded|AgeImputed|AgeVector|           AgeScaled|FareImputed|FareVector|          FareScaled|            Features|
+-----------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+--------+-----------+-------------+----------+---------+--------------------+-----------+----------+--------------------+--------------------+
|        892|     3|    Kelly, Mr. James|  male|34.5|    0|    0| 330911| 7.8292| NULL|       Q|     0.0|        0.0|(4,[0],[1.0])|      34.5|   [34.5]|[0.4282483035938678]|     7.8292|  [7.8292]|[0.3388647951454714]|(7,[1,5,6],[1.0,0...|
|        893|     3|Wilkes, Mrs. Jame...|fem

In [63]:
feat_df.head()[-1]

SparseVector(7, {1: 1.0, 5: 0.2712, 6: 0.3138})